# Week 8 — Capstone evidence and release gate

Graduate a grounded learning assistant with one least-privilege read-only tool. A polished demo is not the release decision: the evidence pack, safety results, ownership, observability, cost, and rollback must pass together.

In [ ]:
import importlib.util
import sys
from pathlib import Path

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
labs = helpers.load_offline_labs(curriculum_root)
session.safe_summary()

In [ ]:
def evidence(status, artifact, owner="group:ai-platform-owners"):
    return labs.EvidenceItem(status, artifact, owner)


required_evidence = (
    "impact_and_ownership",
    "model_comparison",
    "output_boundary",
    "retrieval_security",
    "tool_security",
    "versioned_evaluation",
    "human_safety_review",
    "trace_privacy",
    "failure_drills",
    "cost_and_slos",
    "rollback",
    "runbook",
)

In [ ]:
evidence_map = {
    "impact_and_ownership": evidence("pass", "lesson-00:risk-evidence"),
    "model_comparison": evidence("pass", "lesson-01:model-results"),
    "output_boundary": evidence("pass", "lesson-02:structured-results"),
    "retrieval_security": evidence("pass", "lesson-03:retrieval-evidence"),
    "tool_security": evidence("pass", "lesson-04:tool-evidence"),
    "versioned_evaluation": evidence("pass", "lesson-05:evaluation-evidence"),
    "human_safety_review": evidence("missing", None),
    "trace_privacy": evidence("pass", "lesson-06:trace-field-audit"),
    "failure_drills": evidence("pass", "lesson-06:failure-drills"),
    "cost_and_slos": evidence("missing", None),
    "rollback": evidence("pass", "lesson-06:rollback-proof"),
    "runbook": evidence("missing", None),
}

In [ ]:
release_decision = labs.decide_evidence_map(evidence_map, required_evidence)
completed_fixture = {
    **evidence_map,
    "human_safety_review": evidence("pass", "fixture://human-review"),
    "cost_and_slos": evidence("pass", "fixture://cost-slo-review"),
    "runbook": evidence("pass", "fixture://runbook"),
}
complete_fixture_decision = labs.decide_evidence_map(
    completed_fixture, required_evidence
)
failed_fixture = {
    **completed_fixture,
    "retrieval_security": evidence("fail", "fixture://retrieval-failure"),
}
failed_fixture_decision = labs.decide_evidence_map(failed_fixture, required_evidence)
assert release_decision.decision == "inconclusive"
assert complete_fixture_decision.decision == "adopt"
assert failed_fixture_decision.decision == "reject"

In [ ]:
capstone_evidence = {
    name: {
        "status": item.status,
        "artifact": item.artifact,
        "owner": item.owner,
    }
    for name, item in evidence_map.items()
}
assert set(capstone_evidence) == set(required_evidence)
{
    "evidence_source": labs.EVIDENCE_SOURCE,
    "evidence_map": capstone_evidence,
    "decision": release_decision,
    "note": "offline gate behavior is not production approval",
}

## Required demonstration

1. Answer a grounded question with a resolvable citation.
2. Refuse or safely handle an unsupported question.
3. Deny an unauthorized retrieval attempt.
4. Neutralize indirect prompt injection in retrieved content.
5. Execute one authorized read-only tool call and block a side effect.
6. Show a trace with approved redaction and correlation to the evaluation run.
7. Simulate a dependency failure and demonstrate degraded mode.
8. Roll back to a known-good immutable version.

## Graduation

Twenty cases are a learning minimum, not production proof. A real deployment expands the dataset according to impact, users, languages, failure modes, and regulatory context. Record `adopt`, `reject`, or `inconclusive`; never convert missing evidence into an adoption decision.